# 1.📦 Importación de Librerías 📥📚
---
1. Traigo librerías para empezar,datos y scraping para conectar.
2. Sesión de requests lista quedó,y un plan de reintentos se definió
3. Si falla el Envío por un error,tres veces insiste con gran valor.
4. Se monta el adaptador con atención,aplicando la regla a la conexión.
5. Luego de que revisa, un mensaje avisa que todo cargó con prisa.
***

In [ ]:
# 1.1 Importacion de librerias necesarias
import requests
from bs4 import BeautifulSoup
import sqlite3
import time
import json
import re
from urllib.parse import urljoin,urlparse
from datetime import datetime
import pandas as pd
import os
from typing import Dict,List,Optional,Tuple,Any
import hashlib

#1.2 para manejar errores en red 
from request.adapters import HTTPAdapter
from urllib3.util.retry import Retry

#1.3 Configuracion de sesion con reintentos
session = requests.Session()
retry=Retry(total=3, backoff_factor=1, status_forcelist=[429,500,502,503,504])

#1.4 adaptador
adaptador=HTTPAdapter(max_retries=retry)
session.mount('https://', adaptador)

#1.5crear carpetas datos si no existe
os.makedirs('datos', exist_ok=True)

#1.6 aviso de librerias importadas correctamente 
print("Configuracion completada. librerias importadas")

# 2. 🛠️Funciones auxiliares (🌐HTML, ⏳espera, 💾caché)
1. Obtiene el contenido html de una pagina web, una url y lo devuelve, un objeto beutifulsoup para poder analizarlo.
2. Pausa la ejecución por unos segundos respeta límites de velocidad y no sobrecargar el servidor.
3. Crea un diccionario en memoria para guardar información de autores y evitar repetir llamadas a la API.

In [ ]:
#2.1 Obtiene el contenido html de una pagina web, una url y lo devuelve, un objeto beutifulsoup.
def obtener_html(url:str, timeout:int=10)->Optional[BeautifulSoup]:
    try:
        respuesta=sesion.get(url,timeout=timeout)
        respuesta.raise_for_status()
        soup=BeautifulSoup(respuesta.text,'html.parser')
        print(f"Contenido HTML obtenido correctamente de {url}")
        return soup
    except requests.exceptions.HTTPError as e:
        if e.response.status_code==404:
            print(f"⚠️ Página web no encontrada (404): {url}")
        else:
            print(f"Error HTTP Al obtener el contenido HTML {e.response.status_code} de: {url}")
    except requests.exceptions.Timeout:
        print(f"Tiempo de espera agotado al obtener el contenido HTML de: {url}")
    except requests.exceptions.RequestException as e:
        print(f" Error de red al obtener el contenido HTML de: {url}. Detalles: {e}")

#2.2 funcion para esperar un tiempo pausa para respetar limites de velocidad
def esperar(segundos: int =1):
    print(f"⏳ Esperando {segundos} segundos...")
    time.sleep(segundos)

#2.3 cache de autores en (memoria)
cache_autores={}
